# Movie corpus and exploratory analysis

Audit the source, rebuild the corpus, and inspect revenue coverage and genre trends before modeling. One document is one movie overview.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, StrMethodFormatter
import pandas as pd

from corpus import CLEAN_PATH, RAW_PATH, build_corpus

FIGURES = Path("../outputs")
FIGURES.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#444444",
    "axes.linewidth": 0.6,
    "axes.axisbelow": True,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": "#dddddd",
    "grid.linewidth": 0.5,
    "legend.frameon": False,
    "legend.fontsize": 8,
    "lines.linewidth": 1.4,
    "lines.markersize": 3.6,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

## Source audit

In [ ]:
raw = pd.read_csv(RAW_PATH, low_memory=False)
print(f"{len(raw):,} movies and {raw.shape[1]} fields in the raw CSV")

audit = {}
for field in ("id", "revenue", "budget", "runtime"):
    values = pd.to_numeric(raw[field], errors="coerce")
    audit[field] = {
        "Missing": raw[field].isna().sum(),
        "Malformed": (raw[field].notna() & values.isna()).sum(),
        "Zero": values.eq(0).sum(),
    }
audit_table = pd.DataFrame.from_dict(audit, orient="index")
display(audit_table)

dates = pd.to_datetime(raw["release_date"], format="%Y-%m-%d", errors="coerce")
bad_dates = raw["release_date"].notna() & dates.isna()
repeated_ids = raw["id"].duplicated(keep=False)

print(f"Unparseable release dates: {bad_dates.sum():,}")
print(f"Rows with repeated source IDs: {repeated_ids.sum():,}")

## Build the corpus

The CLI and notebook use the same cleaner. Revenue, budget, and runtime values of zero are treated as missing. The supervised cohort requires positive recorded revenue and a valid release date.

In [ ]:
cleaning_counts = build_corpus()
cleaning_counts.to_frame("Films")

In [ ]:
movies = pd.read_csv(CLEAN_PATH, parse_dates=["release_date"])
labeled = movies.dropna(subset=["revenue", "release_date"]).copy()

fields = ["overview", "release_date", "revenue", "budget", "collection_id"]
coverage = pd.DataFrame({
    "Text corpus": movies[fields].count(),
    "Supervised": labeled[fields].count(),
})
display(coverage)

word_counts = labeled["overview"].str.split().str.len()
length_summary = word_counts.quantile([0.1, 0.5, 0.9]).astype(int)
print(f"Overview length (10th, 50th, 90th percentiles): {length_summary.tolist()} words")

repeated = movies["overview"].str.casefold().duplicated(keep=False)
print(f"Text corpus rows with repeated overviews: {repeated.sum():,}")

## Coverage over time

Yearly counts show where the text corpus and supervised sample differ. Genre plots use the five most common genres among pre-2014 labeled films. Movies can count in multiple genres.

In [ ]:
coverage = (
    movies.assign(has_revenue=movies["revenue"].notna())
    .groupby("release_year")["has_revenue"]
    .agg(Overviews="size", Labeled="sum")
)

fig, ax = plt.subplots(figsize=(6.6, 2.7), layout="constrained")
ax.plot(coverage.index, coverage["Overviews"], color="#245778", label="All overviews")
ax.plot(
    coverage.index, coverage["Labeled"],
    color="#a64c27", linestyle="--", label="With recorded revenue",
)
ax.set(xlabel="Release year", ylabel="Films")
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.legend(loc="upper left")
fig.savefig(FIGURES / "release_year_coverage.png")
plt.show()

## Genre and revenue by release decade

CPI-adjusted medians assume comparable nominal USD. The source does not identify each film's currency or gross scope. The 2010s are only partially covered, and medians are shown when at least 30 films have adjusted revenue.

In [ ]:
genre_movies = labeled[["genres", "release_year", "revenue_cpi_2016"]].copy()
genre_movies["genre"] = genre_movies["genres"].dropna().map(json.loads)
genre_movies["decade"] = genre_movies["release_year"].astype(int) // 10 * 10
genre_movies = genre_movies.explode("genre").dropna(subset=["genre"])

earlier_films = genre_movies.loc[genre_movies["release_year"].lt(2014)]
major_genres = earlier_films["genre"].value_counts().head(5).index

by_genre = genre_movies.groupby(["decade", "genre"])["revenue_cpi_2016"]
counts = by_genre.size().unstack()[major_genres]
enough_films = by_genre.count().ge(30)
median = by_genre.median().where(enough_films).unstack()[major_genres] / 1e6

In [ ]:
fig, (ax_count, ax_revenue) = plt.subplots(
    1, 2, figsize=(6.6, 3.0), sharex=True, layout="constrained",
)

styles = [
    ("#245778", "o", "-"),
    ("#a64c27", "s", "--"),
    ("#33735a", "^", "-."),
    ("#74558d", "D", ":"),
    ("#626262", "v", "--"),
]

for genre, (color, marker, linestyle) in zip(major_genres, styles):
    style = {"color": color, "marker": marker, "linestyle": linestyle}
    ax_count.plot(counts.index, counts[genre], label=genre, **style)
    ax_revenue.plot(median.index, median[genre], **style)

ax_count.set(title="(a) Labeled films", xlabel="Release decade", ylabel="Films")
ax_count.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax_revenue.set(
    title="(b) Median adjusted revenue",
    xlabel="Release decade",
    ylabel="Revenue (millions, 2016 basis)",
    yscale="log",
)
ax_revenue.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:g}"))
ax_count.set_xticks(range(1920, 2020, 20))
handles, labels = ax_count.get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncols=5)
fig.savefig(FIGURES / "genre_by_decade.png")
plt.show()

## Coverage for additional model comparisons

In [ ]:
has_production = (
    labeled[["production_countries", "production_companies"]]
    .fillna("[]")
    .ne("[]")
    .all(axis=1)
)
has_budget = labeled["budget"].notna()
has_collection = labeled["collection_id"].notna()

display(pd.Series({
    "Labeled films": len(labeled),
    "Genres listed": labeled["genres"].fillna("[]").ne("[]").sum(),
    "Production country and company listed": has_production.sum(),
    "Collection listed": has_collection.sum(),
    "Positive budget": has_budget.sum(),
    "Production, collection, and budget": (has_production & has_collection & has_budget).sum(),
    "CPI matched": labeled["revenue_cpi_2016"].notna().sum(),
}, name="Films").to_frame())